# Deploy Claude Code on GCP via Vertex AI (Self-Contained)

This notebook deploys the full reference architecture **without cloning
any repository**. All source code (Dockerfiles, Python services, HTML)
is embedded directly in the cells below.

**What you'll get:**

- An **LLM Gateway** (Cloud Run) that routes Claude Code traffic through Vertex AI
- An **MCP Gateway** (Cloud Run) hosting shared MCP tools
- A **Dev Portal** (Cloud Run) with per-OS setup instructions
- *(optional)* A **Dev VM** (GCE) accessible via IAP TCP tunnel

**Prerequisites:**

- You're authenticated to Google (Colab does this automatically)
- You have `Owner` or `Editor` on the target GCP project
- Billing is enabled on the project
- Claude models are enabled in the Vertex AI Model Garden

> Run the cells top-to-bottom. Each step explains what it does.
> Nothing is created until you explicitly confirm.

## 1. Choose deployment settings

Edit the variables below, then run the cell.

In [ ]:
# ── Edit these for your deployment ──────────────────────────────────────
PROJECT_ID        = 'my-gcp-project'          # REQUIRED — your GCP project ID
REGION            = 'global'                  # Vertex AI region
FALLBACK_REGION   = 'us-central1'             # Cloud Run / GCE region

ENABLE_LLM        = True                      # LLM gateway (reverse proxy to Vertex)
ENABLE_MCP        = True                      # MCP gateway (shared tools)
ENABLE_PORTAL     = True                      # Dev portal (static setup page)
ENABLE_DEV_VM     = False                     # Dev VM (costs money when running)

# Comma-separated IAM members who can invoke the gateways.
ALLOWED_PRINCIPALS = 'user:you@example.com'

# ── Usually no need to change below this line ─────────────────────────────
AR_REPO           = 'claude-code-vertex-gcp'  # Artifact Registry repo name

print(f'Settings captured. Project: {PROJECT_ID}, Region: {REGION}')

## 2. Authenticate and set up helpers

In [ ]:
import os, subprocess, sys, tempfile, shutil
from datetime import datetime, timezone

# ── Shell helper ───────────────────────────────────────────────────────────
def sh(cmd: str, check: bool = True) -> str:
    """Run a shell command, print it, stream output, return stdout."""
    print(f'$ {cmd}')
    proc = subprocess.run(cmd, shell=True, check=check,
                          capture_output=True, text=True)
    if proc.stdout.strip():
        print(proc.stdout)
    if proc.stderr.strip():
        print(proc.stderr, file=sys.stderr)
    return proc.stdout

# ── File writer ────────────────────────────────────────────────────────────
def write_file(path: str, content: str) -> None:
    """Create parent dirs and write content to path."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        f.write(content)

# ── Cloud Run URL lookup ──────────────────────────────────────────────────
def get_service_url(service_name: str) -> str:
    """Query Cloud Run for a service URL. Returns '' if not found."""
    try:
        out = subprocess.check_output(
            ['gcloud', 'run', 'services', 'describe', service_name,
             '--project', PROJECT_ID, '--region', CR_REGION,
             '--format=value(status.url)'],
            text=True, stderr=subprocess.DEVNULL).strip()
        return out
    except subprocess.CalledProcessError:
        return ''

# ── Colab auth ─────────────────────────────────────────────────────────────
try:
    from google.colab import auth as _colab_auth
    _colab_auth.authenticate_user()
    print('Colab auth ok')
except Exception as e:
    print('Not in Colab, skipping colab auth:', e)

# ── Set up environment ─────────────────────────────────────────────────────
sh(f'gcloud config set project {PROJECT_ID}')
sh('gcloud auth list')

CR_REGION = FALLBACK_REGION
IMAGE_TAG = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
WORK_DIR  = tempfile.mkdtemp(prefix='claude-code-deploy-')

os.environ['PROJECT_ID']      = PROJECT_ID
os.environ['REGION']          = REGION
os.environ['FALLBACK_REGION'] = FALLBACK_REGION
os.environ['PRINCIPALS']      = ALLOWED_PRINCIPALS

print(f'Work directory: {WORK_DIR}')
print(f'Image tag:      {IMAGE_TAG}')

In [ ]:
REQUIRED_APIS = [
    'aiplatform.googleapis.com',       'run.googleapis.com',
    'compute.googleapis.com',          'iap.googleapis.com',
    'artifactregistry.googleapis.com', 'cloudbuild.googleapis.com',
    'logging.googleapis.com',          'monitoring.googleapis.com',
    'iamcredentials.googleapis.com',   'secretmanager.googleapis.com',
    'serviceusage.googleapis.com',     'bigquery.googleapis.com',
]
sh(f'gcloud services enable {" ".join(REQUIRED_APIS)} --project {PROJECT_ID}')

## 3. Confirm before creating resources

Everything after this cell may incur GCP charges.

In [ ]:
reply = input(f'Create resources in project {PROJECT_ID}? [yes/NO]: ')
assert reply.strip().lower() in ('y', 'yes'), 'Aborted.'
print('Proceeding.')

In [ ]:
# Create Artifact Registry repository (idempotent)
exists = subprocess.run(
    f'gcloud artifacts repositories describe {AR_REPO} '
    f'--project {PROJECT_ID} --location {CR_REGION}',
    shell=True, capture_output=True).returncode == 0

if not exists:
    sh(f'gcloud artifacts repositories create {AR_REPO} '
       f'--project {PROJECT_ID} --location {CR_REGION} '
       f'--repository-format=docker '
       f'--description="Claude Code on Vertex container images"')
    print(f'Created Artifact Registry repo: {AR_REPO}')
else:
    print(f'Artifact Registry repo {AR_REPO} already exists.')

## 4. Deploy the LLM Gateway

The LLM Gateway is a FastAPI reverse proxy that sits between Claude Code
and Vertex AI. It authenticates callers via Cloud Run IAM, strips
Anthropic-only beta headers, and forwards requests using the gateway's
own service-account credentials.

Build time: ~3-5 minutes.

In [ ]:
if ENABLE_LLM:
    GW = f'{WORK_DIR}/gateway'

    write_file(f'{GW}/Dockerfile', '''# syntax=docker/dockerfile:1.7
# =============================================================================
# LLM Gateway — production container image.
#
# Multi-stage build:
#   * builder: installs dependencies into a venv
#   * runtime: copies only the venv + app, runs as a non-root user
#
# The result is small (~120 MB) and has no build toolchain in the final
# image. Compatible with Cloud Run's buildpacks cache.
# =============================================================================

# -----------------------------------------------------------------------------
# Builder stage.
# We use the full python image (not slim) here because some transitive deps
# need build tooling. The result is thrown away.
# -----------------------------------------------------------------------------
FROM python:3.12-slim AS builder

# Don't write .pyc files, don't buffer stdout. Helps keep layers small and
# makes Cloud Logging see log lines in real time during startup.
ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1 \\
    PIP_DISABLE_PIP_VERSION_CHECK=1 \\
    PIP_NO_CACHE_DIR=1

# Create a venv so the final image can copy a single directory.
RUN python -m venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

WORKDIR /build
COPY requirements.txt ./

# Install exactly what's in requirements.txt. No extras, no dev deps.
RUN pip install --upgrade pip && pip install -r requirements.txt


# -----------------------------------------------------------------------------
# Runtime stage.
# -----------------------------------------------------------------------------
FROM python:3.12-slim AS runtime

ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1 \\
    PATH="/opt/venv/bin:$PATH" \\
    # Tell FastAPI/uvicorn which module to import. Cloud Run passes $PORT
    # at runtime; the CMD below honors it.
    PORT=8080

# Run as a non-root user. Cloud Run does not require this, but it is
# defense-in-depth in case of a sandbox escape.
RUN groupadd --system app && useradd --system --gid app --home /app app
WORKDIR /app

# Copy the prebuilt venv from the builder stage.
COPY --from=builder /opt/venv /opt/venv

# Copy the application source. Ownership assigned to the unprivileged user.
COPY --chown=app:app app ./app

USER app

EXPOSE 8080

# Uvicorn entrypoint. Workers=1 because Cloud Run scales horizontally by
# spawning more container instances; multiple workers inside one container
# would only add memory pressure.
CMD ["sh", "-c", "uvicorn app.main:app --host 0.0.0.0 --port ${PORT} --workers 1"]
''')

    write_file(f'{GW}/requirements.txt', '''# -----------------------------------------------------------------------------
# Runtime dependencies for the LLM gateway Cloud Run service.
#
# Pinned to minor versions. Bump deliberately when upgrading; CI on your
# fork should run the tests in tests/ against the new pins.
# -----------------------------------------------------------------------------

# FastAPI is the web framework. Starlette (its foundation) is what
# actually provides StreamingResponse, which we rely on for streaming
# Vertex responses back to the client.
fastapi>=0.115,<0.120

# Uvicorn is the ASGI server. We run it with one worker per Cloud Run
# container instance; Cloud Run handles horizontal scaling.
uvicorn[standard]>=0.32,<0.40

# httpx is our outbound HTTP client with first-class async support and
# connection pooling. AsyncClient lets us share connections between
# concurrent proxied requests on the same container.
httpx>=0.27,<0.30

# google-auth gives us Application Default Credentials discovery + the
# token-refresh transport. On Cloud Run this automatically picks up the
# service-account attached to the service.
google-auth>=2.34,<3.0

# Kept as a direct dep so a future op can swap to the proper Cloud
# Logging handler without editing requirements. Currently unused at
# runtime (we emit JSON to stdout), but cheap to ship.
google-cloud-logging>=3.11,<4.0
''')

    write_file(f'{GW}/app/__init__.py', '''"""LLM Gateway package.

A tiny FastAPI reverse proxy that sits between Claude Code (running on a
developer's machine or the Dev VM) and the Vertex AI Anthropic endpoint.

It does four things:

1. Accepts incoming HTTPS requests from Claude Code.
2. Verifies (via Cloud Run's built-in IAM enforcement) that the caller has
   ``roles/run.invoker``. Cloud Run rejects unauthenticated callers before
   we ever see them, so by the time a request hits our FastAPI app it is
   already authenticated. We just capture *who* the caller is for logging.
3. Strips Anthropic-only beta headers that Vertex does not accept.
4. Forwards the request to the real Vertex AI endpoint using the Cloud Run
   service account's Application Default Credentials.

See the repo-root ARCHITECTURE.md for the big picture.
"""

# Semantic version of the gateway image. Bump when you change behavior.
__version__ = "0.1.0"
''')

    write_file(f'{GW}/app/main.py', '''"""FastAPI application entrypoint.

This module wires the pieces together:

* Configures JSON logging to stdout (Cloud Logging picks it up).
* Owns the shared ``httpx.AsyncClient`` (one per container instance).
* Defines a catch-all route that forwards every request through the proxy
  logic in :mod:`proxy`.
* Exposes a ``/healthz`` endpoint for Cloud Run's startup/liveness probe.

Why a single catch-all route? Because the gateway is a pass-through. We
don't want to maintain a list of Vertex API paths — Claude Code will use
whichever path is correct for its current code, and we want to forward
anything starting with ``/v1``.
"""

from __future__ import annotations

import os
from contextlib import asynccontextmanager
from typing import AsyncIterator

import httpx
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

from . import __version__
from .logging_config import configure_logging, get_logger
from .proxy import DEFAULT_TIMEOUT, proxy_request


# Configure logging before anything else; the logger is used below.
configure_logging()
log = get_logger(__name__)


# --- Application lifespan ---------------------------------------------------


@asynccontextmanager
async def lifespan(app: FastAPI) -> AsyncIterator[None]:
    """Create and dispose the shared httpx client.

    Called once when the container starts, once when it stops. We attach
    the client to ``app.state`` so every request handler can reach it
    without a module-level global that would interfere with testing.
    """
    log.info("gateway_starting")
    # ``http2=False`` keeps things simple. Vertex supports HTTP/2 but
    # enabling it adds a dependency on the ``h2`` package and offers no
    # measurable latency win for this workload.
    async with httpx.AsyncClient(timeout=DEFAULT_TIMEOUT, http2=False) as client:
        app.state.http_client = client
        yield
    log.info("gateway_stopped")


app = FastAPI(
    title="Claude Code → Vertex AI Gateway",
    version="0.1.0",
    # Disable auto-docs in production: this is a pass-through, not a
    # user-facing API. Cuts the attack surface by one endpoint.
    docs_url=None,
    redoc_url=None,
    openapi_url=None,
    lifespan=lifespan,
)


# --- Routes -----------------------------------------------------------------


@app.get("/healthz")
async def healthz() -> JSONResponse:
    """Cloud Run liveness / readiness probe (legacy path).

    Returns a trivial JSON body with 200. We deliberately do not call
    Vertex here — if Vertex is down we still want the gateway marked
    healthy so it can return the real upstream error to callers.
    """
    return JSONResponse({"status": "ok"})


# The richer /health endpoint is what the end-to-end test script and
# in-VPC monitors use. Two-layer auth picture:
#
#   * **App layer (FastAPI)** — unauthenticated. The handler runs no
#     token validation and reveals nothing sensitive (name + version).
#   * **Platform layer (Cloud Run)** — the service is deployed with
#     ``--no-allow-unauthenticated`` + ``ingress=internal-and-cloud-
#     load-balancing``, so Cloud Run rejects callers lacking
#     ``roles/run.invoker`` BEFORE this handler runs. An anonymous
#     external probe therefore cannot reach this endpoint, regardless
#     of what FastAPI does.
#
# For Cloud Monitoring uptime checks: create a service account, grant
# ``roles/run.invoker`` on the gateway, and configure the uptime check
# to sign requests with that SA. See README.md → "External uptime
# probes".
#
# FastAPI routes are matched in declaration order; the catch-all
# ``/{full:path}`` route declared BELOW this one only matches when no
# explicit route matches, so /health does not fall through to the
# proxy.
@app.get("/health")
async def health() -> JSONResponse:
    """Liveness endpoint. Cloud Run IAM gates access; app layer is open."""
    return JSONResponse(
        {
            "status": "ok",
            "component": "llm_gateway",
            # Version comes from the package __init__; overridable via
            # GATEWAY_VERSION env for ops who pin images to a git SHA.
            "version": os.getenv("GATEWAY_VERSION", __version__),
        }
    )


# The catch-all route. FastAPI's path converter ``:path`` means ``full``
# captures everything including slashes. The explicit methods list
# includes POST (predict calls) and GET (rare but possible for discovery).
@app.api_route(
    "/{full:path}",
    methods=["GET", "POST", "PUT", "DELETE", "PATCH", "OPTIONS"],
)
async def proxy(full: str, request: Request):
    """Forward any request to Vertex AI.

    The ``full`` argument is unused directly — we work off
    ``request.url.path`` which is the whole inbound path. We keep ``full``
    in the signature so FastAPI knows this route accepts any path.
    """
    client: httpx.AsyncClient = request.app.state.http_client
    try:
        return await proxy_request(request, client)
    except httpx.HTTPError as exc:
        # Upstream network problem. Log it with caller context and return
        # a 502, which is the semantically correct code for "I am a proxy
        # and my upstream is broken."
        log.exception(
            "upstream_error",
            extra={"path": request.url.path, "error_type": type(exc).__name__},
        )
        return JSONResponse(
            {"error": "upstream_unavailable", "detail": str(exc)},
            status_code=502,
        )
''')

    write_file(f'{GW}/app/proxy.py', '''"""Pass-through reverse proxy logic.

This is the core of the gateway. Given an incoming request from Claude
Code, we:

1. Compute the Vertex AI upstream URL (same path, swapped hostname).
2. Sanitize the request headers (strip Anthropic-only betas, etc.).
3. Add an ``Authorization: Bearer <token>`` header using the gateway's
   own service-account credentials — the caller's token is NOT forwarded.
4. Stream the request body through to Vertex, stream the response back.
5. Emit a structured log entry with caller, model, status, and latency.

We do **not** rewrite payloads. Claude Code emits Vertex-formatted JSON
when ``CLAUDE_CODE_USE_VERTEX=1``, and we pass it through unchanged.
"""

from __future__ import annotations

import os
import re
import time
from typing import AsyncIterator, Optional

import httpx
from fastapi import Request
from fastapi.responses import StreamingResponse

from .auth import extract_caller_identity, get_vertex_access_token
from .headers import sanitize_request_headers
from .logging_config import get_logger

log = get_logger(__name__)


# The Cloud Run region of this service. Used only for logging; has nothing
# to do with the Vertex region we call. Set automatically by Cloud Run.
_CLOUD_RUN_REGION = os.getenv("K_SERVICE_REGION") or os.getenv("GOOGLE_CLOUD_REGION", "unknown")

# Default Vertex region if the caller did not encode one in the URL path.
# "global" is the multi-region Vertex endpoint, which uses a different
# hostname from regional endpoints (see ``_vertex_host_for_region``).
_DEFAULT_VERTEX_REGION = os.getenv("VERTEX_DEFAULT_REGION", "global")

# The GCP project ID for Vertex calls. Cloud Run populates GOOGLE_CLOUD_PROJECT.
# Falls back to VERTEX_PROJECT_ID if the operator set it explicitly.
_VERTEX_PROJECT_ID = (
    os.getenv("VERTEX_PROJECT_ID")
    or os.getenv("GOOGLE_CLOUD_PROJECT")
    or ""
)


# Shared httpx client. Creating one per request would defeat connection
# pooling and keepalive. Created in main.py's lifespan hook and injected
# into functions here via the request state.
#
# The timeout is generous because Claude responses (especially with long
# contexts) can take tens of seconds. We rely on Cloud Run's own 15-minute
# request timeout as the outer bound.
DEFAULT_TIMEOUT = httpx.Timeout(
    connect=10.0,
    read=300.0,
    write=60.0,
    pool=10.0,
)


def _vertex_host_for_region(region: str) -> str:
    """Return the correct Vertex AI hostname for a given region.

    The multi-region "global" endpoint is served from the bare
    ``aiplatform.googleapis.com`` hostname. Every specific region has its
    own regional hostname.

    Args:
        region: A Vertex region string, e.g. "us-east5" or "global".

    Returns:
        Hostname suitable for use in an HTTPS URL.
    """
    if region == "global":
        return "aiplatform.googleapis.com"
    return f"{region}-aiplatform.googleapis.com"


# Paths that look like Vertex publisher calls. We use this to extract the
# model name for logging, e.g. ``.../publishers/anthropic/models/claude-opus-4-6:rawPredict``.
_MODEL_IN_PATH = re.compile(r"/publishers/anthropic/models/([^:/]+)")


def _extract_model(path: str) -> Optional[str]:
    """Pull the Anthropic model name out of a Vertex URL path, if present."""
    match = _MODEL_IN_PATH.search(path)
    return match.group(1) if match else None


def _extract_region_from_path(path: str) -> str:
    """If the URL path starts with ``/v1/projects/<p>/locations/<r>/...``
    pull ``<r>`` out and use it as the Vertex region. Otherwise use the
    service-wide default.
    """
    # The regex is anchored at start-of-path and tolerant of leading
    # slashes. ``/v1beta1/`` is also a valid Vertex API prefix.
    match = re.match(
        r"^/?v\\d[^/]*/projects/[^/]+/locations/([^/]+)/",
        path,
    )
    if match:
        return match.group(1)
    return _DEFAULT_VERTEX_REGION


def build_upstream_url(path: str, query: str) -> str:
    """Compose the full Vertex URL for an inbound path.

    Args:
        path: Request path as seen by FastAPI, e.g.
              ``/v1/projects/my-proj/locations/us-east5/publishers/anthropic/models/claude-opus-4-6:rawPredict``.
        query: Raw query string (no leading ``?``).

    Returns:
        The fully-qualified Vertex URL with the correct regional host.
    """
    region = _extract_region_from_path(path)
    host = _vertex_host_for_region(region)

    # Ensure there is exactly one leading slash on the path portion.
    clean_path = "/" + path.lstrip("/")

    url = f"https://{host}{clean_path}"
    if query:
        url = f"{url}?{query}"
    return url


async def _iter_upstream(response: httpx.Response) -> AsyncIterator[bytes]:
    """Yield the upstream response body chunk by chunk.

    Using an async iterator here lets us stream very large responses
    (Claude with long output + thinking tokens) back to the client without
    buffering the whole thing in memory.
    """
    async for chunk in response.aiter_raw():
        yield chunk


async def proxy_request(
    request: Request,
    client: httpx.AsyncClient,
) -> StreamingResponse:
    """Forward one inbound request to Vertex AI and stream the response back.

    Args:
        request: The FastAPI request. We read body, headers, path, query.
        client: A shared ``httpx.AsyncClient`` (created in app lifespan).

    Returns:
        A ``StreamingResponse`` that proxies the upstream body to the client.
    """
    started = time.monotonic()

    # --- Inspect the inbound request (for logging) -------------------------
    caller = extract_caller_identity(
        {k.lower(): v for k, v in request.headers.items()}
    )
    model = _extract_model(request.url.path)
    region = _extract_region_from_path(request.url.path)

    # --- Sanitize headers and add gateway auth -----------------------------
    cleaned, stripped = sanitize_request_headers(
        [(k, v) for k, v in request.headers.items()]
    )
    cleaned["Authorization"] = f"Bearer {get_vertex_access_token()}"

    # --- Build upstream URL ------------------------------------------------
    url = build_upstream_url(request.url.path, request.url.query)

    # --- Read body ---------------------------------------------------------
    # Claude Code requests are JSON and usually small; we read the whole
    # body into memory. If we ever need to handle multi-megabyte uploads
    # we would switch to streaming the request body instead.
    body = await request.body()

    # --- Forward -----------------------------------------------------------
    # httpx ``stream`` returns an async context manager around a response
    # whose body has not been fully read; we pass that streaming body back
    # to the client, so the client sees Vertex's bytes as they arrive.
    req = client.build_request(
        method=request.method,
        url=url,
        headers=cleaned,
        content=body,
    )
    upstream = await client.send(req, stream=True)

    elapsed_ms = int((time.monotonic() - started) * 1000)
    log.info(
        "proxy_request",
        extra={
            "caller": caller.email,
            "caller_source": caller.source,
            "method": request.method,
            "path": request.url.path,
            "upstream_host": url.split("/", 3)[2],
            "vertex_region": region,
            "model": model,
            "status_code": upstream.status_code,
            "latency_ms_to_headers": elapsed_ms,
            "betas_stripped": [h for h in stripped if h.startswith("anthropic-beta")],
            "cloud_run_region": _CLOUD_RUN_REGION,
            "project_id": _VERTEX_PROJECT_ID,
        },
    )

    # --- Relay response ----------------------------------------------------
    # Strip hop-by-hop headers on the way back too. ``Transfer-Encoding``
    # in particular must not be forwarded, since FastAPI/Starlette will
    # recompute framing.
    response_headers = {
        k: v
        for k, v in upstream.headers.items()
        if k.lower() not in {"content-encoding", "transfer-encoding", "connection"}
    }

    return StreamingResponse(
        _iter_upstream(upstream),
        status_code=upstream.status_code,
        headers=response_headers,
        background=_make_close_task(upstream),
    )


def _make_close_task(upstream: httpx.Response):
    """Wrap the upstream close in a Starlette ``BackgroundTask``.

    Starlette's ``StreamingResponse`` accepts a ``BackgroundTask`` that
    runs after the response body has been sent. We use this to close the
    upstream httpx response, releasing the underlying connection back to
    the pool.
    """
    # Imported lazily because starlette is a transitive dependency of
    # fastapi and we don't want to import at module load if something
    # test-time patches the environment.
    from starlette.background import BackgroundTask

    async def _close() -> None:
        await upstream.aclose()

    return BackgroundTask(_close)
''')

    write_file(f'{GW}/app/auth.py', '''"""Caller-identity extraction.

**Important context on who actually enforces auth:**

Cloud Run, when deployed with ``--no-allow-unauthenticated``, enforces
IAM on the caller *before* the request reaches our FastAPI app. If the
caller does not have ``roles/run.invoker`` on the service, Cloud Run
returns 403 and our code never runs. That means this module does NOT
need to validate signatures or check token audiences — the platform did
that already.

What this module *does* do is:

1. Extract the caller's email from the headers that Cloud Run injects on
   authenticated requests so we can include it in logs.
2. Obtain a fresh OAuth 2.0 access token for the gateway's own service
   account via Application Default Credentials — this is the token we
   will use when calling Vertex AI.

When running locally (outside Cloud Run) for dev testing, the caller
headers are absent, so we fall back to a ``local-dev`` label.
"""

from __future__ import annotations

import threading
from dataclasses import dataclass
from typing import Optional

import google.auth
import google.auth.transport.requests
from google.auth.credentials import Credentials


# Vertex AI OAuth scope — required for any call to aiplatform.googleapis.com.
_VERTEX_SCOPE = "https://www.googleapis.com/auth/cloud-platform"


@dataclass(frozen=True)
class CallerIdentity:
    """Who made the inbound request.

    Attributes:
        email: The caller's email. ``None`` if we couldn't determine it
            (e.g., local dev, or the caller is a service account without
            an injected email header).
        source: Which header family the identity came from. One of
            ``"iap"``, ``"cloud_run"``, or ``"unknown"``. Useful for
            debugging auth flows.
    """
    email: Optional[str]
    source: str


def extract_caller_identity(headers: dict[str, str]) -> CallerIdentity:
    """Identify who made the request.

    Cloud Run injects ``X-Goog-Authenticated-User-Email`` on authenticated
    invocations. IAP injects ``X-Goog-Authenticated-User-Email`` too
    (prefixed with ``accounts.google.com:``). Either way, the header's
    value ends with the email, which is what we care about.

    Args:
        headers: Case-insensitive dict of the inbound request headers.
            Call sites should lower-case header names before lookup, or
            use FastAPI's ``request.headers`` which is already
            case-insensitive.

    Returns:
        A ``CallerIdentity``. ``email`` is ``None`` for local-dev calls.
    """
    # Both IAP and Cloud Run use the same header name; values from IAP are
    # of the form "accounts.google.com:[email protected]", while direct
    # Cloud Run invocations produce the raw email. We handle both.
    raw = headers.get("x-goog-authenticated-user-email")
    if not raw:
        return CallerIdentity(email=None, source="unknown")

    # Strip the IAP prefix if present.
    if ":" in raw:
        email = raw.split(":", 1)[1]
        source = "iap"
    else:
        email = raw
        source = "cloud_run"

    return CallerIdentity(email=email, source=source)


# --- Gateway-side credentials for outbound Vertex calls --------------------

# The Cloud Run runtime populates ADC automatically with the service account
# attached to the service. We cache the credentials object module-wide and
# refresh its token on demand, which is cheap and thread-safe thanks to
# google-auth's internal locking.
_credentials_lock = threading.Lock()
_credentials: Optional[Credentials] = None


def _get_credentials() -> Credentials:
    """Fetch (and cache) Application Default Credentials scoped for Vertex.

    Uses a module-level cache protected by a lock. The credentials object
    itself is safe to share across threads and across async tasks.
    """
    global _credentials
    with _credentials_lock:
        if _credentials is None:
            # ``google.auth.default`` picks up the ambient identity:
            #   * On Cloud Run: the service account attached to the service.
            #   * Locally: whoever ran ``gcloud auth application-default login``.
            creds, _project = google.auth.default(scopes=[_VERTEX_SCOPE])
            _credentials = creds
        return _credentials


def get_vertex_access_token() -> str:
    """Return a fresh OAuth access token for calling Vertex AI.

    Refreshes the cached credentials if the token is missing or expired.
    Call this on every outbound request — ``google-auth`` internally
    avoids network traffic when the existing token is still valid.
    """
    creds = _get_credentials()
    # ``creds.valid`` is False if the token is missing or expired.
    if not creds.valid:
        # ``Request`` is a simple httplib-based transport from google-auth
        # used only for the token refresh endpoint call. It's fine to
        # instantiate per call; it does not hold state between calls.
        request = google.auth.transport.requests.Request()
        creds.refresh(request)
    # ``token`` is the bearer string once the credentials are valid.
    return creds.token  # type: ignore[no-any-return]
''')

    write_file(f'{GW}/app/headers.py', '''"""Header sanitation.

Claude Code occasionally sends ``anthropic-beta`` (and similar) headers to
opt into experimental features. These are supported by Anthropic's direct
API but rejected by Vertex AI, which strict-validates its request headers.
If we forward them as-is, Vertex returns an obscure error and the user's
request fails.

Claude Code exposes ``CLAUDE_CODE_DISABLE_EXPERIMENTAL_BETAS=1`` to
suppress them client-side. ``developer-setup.sh`` sets that flag. This
module is the belt-and-suspenders server-side equivalent: even if someone
runs Claude Code without the flag, we strip the headers here so requests
still succeed.

Also responsible for:
  * Dropping the inbound ``Authorization`` header (the caller's Google ID
    token) — we re-authenticate to Vertex using the Cloud Run service
    account's ADC, not the caller's token.
  * Dropping hop-by-hop headers per RFC 7230 (Connection, Upgrade, etc.)
    that should not be forwarded by any proxy.
"""

from __future__ import annotations

from typing import Iterable


# Exact-match headers we always drop. Compared case-insensitively.
#
# - Authorization: replaced downstream with the gateway SA's token.
# - Host: uvicorn sets this to the Cloud Run hostname; Vertex needs the
#   Vertex hostname, which httpx sets automatically when we build the URL.
# - Content-Length: httpx recomputes this when we rebuild the request body.
# - x-cloud-trace-context / x-goog-*: GCP-internal, stripping is safer.
_EXACT_DROPS = {
    "authorization",
    "host",
    "content-length",
    "x-cloud-trace-context",
    "x-forwarded-for",
    "x-forwarded-proto",
    "x-forwarded-host",
    "forwarded",
}

# Hop-by-hop headers per RFC 7230 Section 6.1. Proxies MUST NOT forward
# these.
_HOP_BY_HOP = {
    "connection",
    "keep-alive",
    "proxy-authenticate",
    "proxy-authorization",
    "te",
    "trailers",
    "transfer-encoding",
    "upgrade",
}

# Prefix-match headers to drop. Any inbound header whose name starts with
# one of these prefixes (case-insensitive) is removed.
#
# "anthropic-beta" — the whole point of this module.
# "x-goog-"        — Cloud Run / IAP internal headers that Vertex doesn't
#                    care about and that can leak caller info we'd rather
#                    re-inject deliberately in the proxy layer.
_PREFIX_DROPS = (
    "anthropic-beta",
    "x-goog-",
)


def sanitize_request_headers(
    headers: Iterable[tuple[str, str]],
) -> tuple[dict[str, str], list[str]]:
    """Return a cleaned copy of the headers suitable for forwarding.

    Args:
        headers: Iterable of (name, value) pairs as they arrived. FastAPI's
            ``request.headers.raw`` returns bytes; callers should decode
            before passing in.

    Returns:
        A 2-tuple ``(clean_headers, stripped_names)`` where:
          * ``clean_headers`` is a dict of header name → value, ready to
            pass to httpx.
          * ``stripped_names`` is a list of lowercase header names that
            were removed (for logging).
    """
    clean: dict[str, str] = {}
    stripped: list[str] = []

    for name, value in headers:
        lower = name.lower()

        # Exact-drop set.
        if lower in _EXACT_DROPS or lower in _HOP_BY_HOP:
            stripped.append(lower)
            continue

        # Prefix-drop set.
        if any(lower.startswith(p) for p in _PREFIX_DROPS):
            stripped.append(lower)
            continue

        # Everything else is safe to forward. Use the original-case name so
        # downstream tooling (e.g., httpx debug output) looks normal.
        clean[name] = value

    return clean, stripped
''')

    write_file(f'{GW}/app/logging_config.py', '''"""Structured JSON logging for Cloud Logging.

Cloud Logging auto-parses JSON on stdout from Cloud Run containers. Any
top-level field we emit becomes a structured, queryable field in Cloud
Logging. This lets the Looker Studio dashboard filter by caller, model,
status, latency, etc. without any log-parsing regex.

Why not use ``google-cloud-logging``'s Python handler directly? Because on
Cloud Run the simplest and cheapest path is: write JSON to stdout, let the
platform pick it up. No extra network calls, no extra dependencies beyond
the stdlib ``logging`` module.

Usage::

    from .logging_config import configure_logging, get_logger

    configure_logging()
    log = get_logger(__name__)
    log.info("request", extra={"caller": "[email protected]", "model": "opus"})
"""

from __future__ import annotations

import json
import logging
import sys
from typing import Any


# Map Python log levels to Google Cloud Logging severities. Cloud Logging
# looks for a top-level "severity" field; if we emit the right value here,
# the log entry shows the correct severity color and icon in the UI.
_SEVERITY = {
    logging.DEBUG: "DEBUG",
    logging.INFO: "INFO",
    logging.WARNING: "WARNING",
    logging.ERROR: "ERROR",
    logging.CRITICAL: "CRITICAL",
}

# Attributes that stdlib ``logging.LogRecord`` sets for internal bookkeeping
# (filename, threadName, etc.). We skip these during serialization so our
# JSON payload stays focused on semantic fields.
_RESERVED_LOGRECORD_ATTRS = {
    "args", "asctime", "created", "exc_info", "exc_text", "filename",
    "funcName", "levelname", "levelno", "lineno", "message", "module",
    "msecs", "name", "pathname", "process", "processName",
    "relativeCreated", "stack_info", "thread", "threadName", "msg",
    "taskName",
}


class CloudLoggingJsonFormatter(logging.Formatter):
    """Serializes each log record as a single JSON object on one line.

    Anything passed in via ``extra={...}`` on a logger call becomes a
    top-level field, which means you can filter for it in Cloud Logging
    with queries like ``jsonPayload.caller="[email protected]"``.
    """

    def format(self, record: logging.LogRecord) -> str:
        payload: dict[str, Any] = {
            "severity": _SEVERITY.get(record.levelno, "DEFAULT"),
            "message": record.getMessage(),
            "logger": record.name,
        }

        # Include exception text if there is one — Cloud Logging shows this
        # in the expanded entry view.
        if record.exc_info:
            payload["exception"] = self.formatException(record.exc_info)

        # Merge `extra={...}` fields. These arrive on the record as
        # attributes that are not part of the standard reserved set.
        for key, value in record.__dict__.items():
            if key in _RESERVED_LOGRECORD_ATTRS or key.startswith("_"):
                continue
            try:
                json.dumps(value)
                payload[key] = value
            except (TypeError, ValueError):
                # Non-JSON-serializable value — coerce to string so it still
                # lands in the log rather than being dropped silently.
                payload[key] = str(value)

        return json.dumps(payload, ensure_ascii=False)


def configure_logging(level: int = logging.INFO) -> None:
    """Configure the root logger for Cloud Logging-friendly JSON output.

    Call this exactly once at application startup, before any log calls.

    Args:
        level: Minimum log level to emit. Defaults to ``INFO``.
    """
    root = logging.getLogger()
    root.setLevel(level)

    # Replace any existing handlers (e.g., uvicorn's default plaintext
    # handler) with our JSON handler. Otherwise logs would emit twice in
    # two formats, which doubles cost and breaks structured parsing.
    for handler in list(root.handlers):
        root.removeHandler(handler)

    handler = logging.StreamHandler(stream=sys.stdout)
    handler.setFormatter(CloudLoggingJsonFormatter())
    root.addHandler(handler)

    # Route uvicorn's loggers through our root handler so access logs are
    # JSON-formatted too.
    for name in ("uvicorn", "uvicorn.access", "uvicorn.error"):
        lg = logging.getLogger(name)
        lg.handlers = []
        lg.propagate = True


def get_logger(name: str) -> logging.Logger:
    """Return a named logger. Thin wrapper over ``logging.getLogger``."""
    return logging.getLogger(name)
''')

    print(f'LLM Gateway source written to {GW}')
else:
    print('LLM gateway disabled, skipping.')

In [ ]:
if ENABLE_LLM:
    SA_ID = 'llm-gateway'
    SA_EMAIL = f'{SA_ID}@{PROJECT_ID}.iam.gserviceaccount.com'
    SERVICE = 'llm-gateway'
    IMAGE = f'{CR_REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}/{SERVICE}:{IMAGE_TAG}'

    # Create service account (idempotent)
    sa_exists = subprocess.run(
        f'gcloud iam service-accounts describe {SA_EMAIL} --project {PROJECT_ID}',
        shell=True, capture_output=True).returncode == 0
    if not sa_exists:
        sh(f'gcloud iam service-accounts create {SA_ID} '
           f'--project {PROJECT_ID} '
           f'--display-name="LLM Gateway (Claude Code -> Vertex)"')

    # Grant IAM roles to SA
    for role in ['roles/aiplatform.user', 'roles/logging.logWriter']:
        sh(f'gcloud projects add-iam-policy-binding {PROJECT_ID} '
           f'--member="serviceAccount:{SA_EMAIL}" '
           f'--role={role} --condition=None --quiet',
           check=False)

    # Build image
    sh(f'gcloud builds submit {WORK_DIR}/gateway '
       f'--project {PROJECT_ID} --tag {IMAGE} --region {CR_REGION}')

    # Deploy to Cloud Run
    sh(f'gcloud run deploy {SERVICE} '
       f'--project {PROJECT_ID} --region {CR_REGION} '
       f'--image {IMAGE} '
       f'--service-account {SA_EMAIL} '
       f'--ingress internal-and-cloud-load-balancing '
       f'--no-allow-unauthenticated '
       f'--min-instances 0 --max-instances 10 '
       f'--cpu 1 --memory 512Mi --port 8080 '
       f'--set-env-vars GOOGLE_CLOUD_PROJECT={PROJECT_ID},VERTEX_DEFAULT_REGION={REGION} '
       f'--quiet')

    # Grant invoker to principals
    principals = [p.strip() for p in ALLOWED_PRINCIPALS.split(',') if p.strip()]
    for p in principals:
        sh(f'gcloud run services add-iam-policy-binding {SERVICE} '
           f'--project {PROJECT_ID} --region {CR_REGION} '
           f'--member="{p}" --role="roles/run.invoker" --quiet',
           check=False)

    LLM_GATEWAY_URL = get_service_url(SERVICE)
    print(f'\nLLM Gateway deployed: {LLM_GATEWAY_URL}')
else:
    LLM_GATEWAY_URL = ''
    print('LLM gateway disabled, skipping.')

## 5. Deploy the MCP Gateway

The MCP Gateway hosts shared Model Context Protocol tools. Claude Code
connects to it via Streamable HTTP. Ships with one built-in tool
(`gcp_project_info`) as a deployment smoke test.

In [ ]:
if ENABLE_MCP:
    MG = f'{WORK_DIR}/mcp-gateway'

    write_file(f'{MG}/Dockerfile', '''# syntax=docker/dockerfile:1.7
# =============================================================================
# MCP Gateway container image — uv-based build.
#
# Follows the pattern Google publishes for MCP-on-Cloud-Run: use the
# official `uv` image for fast, reproducible dependency installs via a
# lockfile, then copy the resulting venv into a slim runtime image.
#
# Why uv instead of pip?
#   * Orders of magnitude faster on cold builds — matters when your team
#     iterates on the server.
#   * Produces a uv.lock for reproducibility.
#   * This is the pattern Google's own samples use.
# =============================================================================

# -----------------------------------------------------------------------------
# Stage 1: build dependencies with uv.
# -----------------------------------------------------------------------------
FROM ghcr.io/astral-sh/uv:python3.12-bookworm-slim AS builder

ENV UV_COMPILE_BYTECODE=1 \\
    UV_LINK_MODE=copy \\
    # Install into /app/.venv so the runtime image copies a single dir.
    UV_PROJECT_ENVIRONMENT=/app/.venv

WORKDIR /app

# Copy only the files needed to resolve deps first, so Docker can cache
# this layer across code-only changes.
COPY pyproject.toml ./
# The lockfile is optional; if absent, uv will produce one during sync.
# Using a two-file COPY with pyproject.toml (which already exists) ensures
# the command never fails when uv.lock is missing.
COPY pyproject.toml uv.lock* ./

# Pre-fetch dependencies WITHOUT the project source. This keeps the
# dependency layer independent of source changes.
RUN uv sync --no-dev --no-install-project --frozen || uv sync --no-dev --no-install-project

# Now copy the application source and install the project itself.
COPY server.py ./
COPY tools ./tools

RUN uv sync --no-dev --frozen || uv sync --no-dev


# -----------------------------------------------------------------------------
# Stage 2: slim runtime.
# -----------------------------------------------------------------------------
FROM python:3.12-slim AS runtime

ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1 \\
    PATH="/app/.venv/bin:$PATH" \\
    PORT=8080

RUN groupadd --system app && useradd --system --gid app --home /app app
WORKDIR /app

# Copy the pre-built venv and the application.
COPY --from=builder --chown=app:app /app/.venv /app/.venv
COPY --chown=app:app server.py ./
COPY --chown=app:app tools ./tools

USER app
EXPOSE 8080

# FastMCP reads $PORT internally (we pass it in server.py). No shell form
# needed for variable expansion because we handle PORT in Python.
CMD ["python", "server.py"]
''')

    write_file(f'{MG}/pyproject.toml', '''# -----------------------------------------------------------------------------
# pyproject.toml for the MCP gateway.
#
# We use `uv` as the package manager + lockfile tool because it is what the
# official Google MCP-on-Cloud-Run pattern uses, and it's dramatically faster
# than pip for container builds. Compatible with the standard PEP 621
# metadata here, so `pip install .` also works as a fallback.
# -----------------------------------------------------------------------------

[project]
name = "claude-code-gcp-mcp-gateway"
version = "0.1.0"
description = "MCP gateway hosting shared tools for Claude Code on GCP."
readme = "README.md"
requires-python = ">=3.11"
license = { text = "Apache-2.0" }

# Runtime dependencies. Keep this set small — tools that need extra
# libraries (e.g., google-cloud-storage) can add them here.
dependencies = [
  # FastMCP is the MCP server library. Pinned to a recent minor; the
  # Streamable HTTP transport is stable in 0.2+. FastMCP 2.x exposes
  # http_app(); older versions fall back to streamable_http_app() —
  # server.py handles both.
  "fastmcp>=0.2,<3.0",

  # FastAPI + uvicorn host the composition: /health at root, FastMCP
  # mounted at /mcp. FastMCP itself pulls in Starlette already; we list
  # FastAPI explicitly so this dep is obvious.
  "fastapi>=0.115,<0.120",
  "uvicorn[standard]>=0.32,<0.40",

  # google-auth for Application Default Credentials. Used by the example
  # gcp_project_info tool and almost certainly by any real tool you add.
  # requests is required by google.auth.transport.requests.
  "google-auth>=2.34,<3.0",
  "requests>=2.31,<3.0",
]

[project.optional-dependencies]
# Dev extras: pytest for smoke testing a tool in isolation.
dev = [
  "pytest>=8,<9",
]

# `uv` uses the standard [tool.uv] section for its own knobs (index URL,
# sync settings, etc.). We don't need anything custom here today.
''')

    write_file(f'{MG}/server.py', '''"""MCP Gateway — FastMCP server over Streamable HTTP, mounted under FastAPI.

Why FastAPI in front of FastMCP?
  * We want a plain unauthenticated ``GET /health`` for load balancers
    and monitoring, which lives alongside the MCP endpoint.
  * FastMCP's HTTP transport is an ASGI app; FastAPI/Starlette can
    mount it cleanly at ``/mcp`` while keeping our own routes at the
    root.

**Transport.** Streamable HTTP (the March 2025 MCP spec standard). SSE
is deprecated and NOT used here.

**Auth.** Cloud Run enforces ``roles/run.invoker`` on the caller for
the /mcp endpoint (the service is deployed with
``--no-allow-unauthenticated``). The /health endpoint is also reached
through Cloud Run's IAM gate — if you need it accessible from an
external uptime monitor, put it behind an IAP-fronted LB or grant
allUsers roles/run.invoker specifically for the health path (not
supported by Cloud Run today; use a separate service in that case).

**Endpoints**:
  * ``GET  /health``  →  unauthenticated liveness check
  * ``POST /mcp``     →  MCP Streamable HTTP endpoint (plus GET/DELETE
                         for session management per the spec)
"""

from __future__ import annotations

import logging
import os
import sys

from fastapi import FastAPI
from fastapi.responses import JSONResponse

# FastMCP is the Python MCP server library we use. It speaks Streamable
# HTTP out of the box and handles all the protocol framing for us.
from fastmcp import FastMCP

# Local tool implementations. Add new imports here when you add new
# files under tools/.
from tools.gcp_project_info import get_project_info


# ----------------------------------------------------------------------------
# Logging — mirror the LLM gateway's "JSON to stdout" pattern so Cloud
# Logging parses structured fields.
# ----------------------------------------------------------------------------

def _configure_logging() -> None:
    """Configure JSON-ish stdout logging for Cloud Run."""
    logging.basicConfig(
        level=logging.INFO,
        stream=sys.stdout,
        format='{"severity":"%(levelname)s","logger":"%(name)s","message":"%(message)s"}',
    )


_configure_logging()
log = logging.getLogger("mcp-gateway")


# ----------------------------------------------------------------------------
# FastMCP server + tool registration
# ----------------------------------------------------------------------------

# Give the server a stable name — Claude Code uses this to label the
# tools it sees in a session.
mcp = FastMCP("claude-code-gcp-mcp-gateway")


@mcp.tool()
def gcp_project_info() -> dict:
    """Return metadata about the GCP project this MCP gateway runs in.

    Useful as a sanity check after deployment: invoke this tool from
    Claude Code and you should see your project ID echoed back.

    Returns:
        A dict with ``project_id``, ``project_number``, ``region``, and
        ``enabled_apis``. Errors surface as ``error`` / ``warning``
        fields rather than raised exceptions.
    """
    log.info("tool_call gcp_project_info")
    return get_project_info()


# ----------------------------------------------------------------------------
# Build the Streamable-HTTP ASGI app from FastMCP.
#
# FastMCP's public API for this changed across versions:
#   * 2.x: mcp.http_app(path="/mcp") — returns a Starlette app
#   * earlier: mcp.streamable_http_app()
# We try the newer method first and fall back. Either way we end up
# with an ASGI app to mount.
# ----------------------------------------------------------------------------

def _build_mcp_asgi():
    """Return a Starlette/ASGI app serving the MCP Streamable HTTP transport."""
    if hasattr(mcp, "http_app"):
        # Newer FastMCP: http_app() accepts a path kwarg and handles
        # lifespan for us. Requesting path="/" so we can mount the
        # result at "/mcp" and get routes at /mcp exactly.
        return mcp.http_app(path="/")
    if hasattr(mcp, "streamable_http_app"):
        return mcp.streamable_http_app()
    raise RuntimeError(
        "This FastMCP version exposes neither http_app() nor "
        "streamable_http_app(); please upgrade fastmcp."
    )


mcp_asgi = _build_mcp_asgi()


# ----------------------------------------------------------------------------
# FastAPI composition.
#
# IMPORTANT: FastMCP relies on an ASGI lifespan to start its internal
# task manager. When we mount FastMCP under FastAPI, we must propagate
# its lifespan context to the parent app — otherwise tool calls will
# fail with "task group not initialized" errors.
# ----------------------------------------------------------------------------

from contextlib import asynccontextmanager


@asynccontextmanager
async def _lifespan(app: FastAPI):
    """Propagate the FastMCP sub-app lifespan into the parent FastAPI app.

    Starlette does not automatically forward lifespan events to mounted
    sub-applications, so we must do it manually. We look for the
    lifespan handler on the sub-app's router and invoke it ourselves.
    """
    _lc = getattr(getattr(mcp_asgi, "router", None), "lifespan_context", None)
    if _lc is not None:
        async with _lc(mcp_asgi):
            yield
    else:
        yield


app = FastAPI(
    title="Claude Code MCP Gateway",
    version="0.1.0",
    docs_url=None,
    redoc_url=None,
    openapi_url=None,
    lifespan=_lifespan,
)


@app.get("/health")
async def health() -> JSONResponse:
    """Liveness probe — app-layer unauth, platform-layer IAM-gated.

    Two-layer auth picture (same as the LLM gateway):

      * **App layer**: this handler does no token validation and
        reveals nothing sensitive.
      * **Platform layer**: Cloud Run enforces ``roles/run.invoker``
        before this handler runs. An anonymous external probe cannot
        reach here. For Cloud Monitoring uptime checks, sign requests
        with a service account that has ``run.invoker`` on this
        service — see README.md → "External uptime probes".

    Returns 200 as long as the process is up; does NOT call into
    FastMCP or any GCP API, so a failing tool does not fail health.
    """
    return JSONResponse({"status": "ok", "component": "mcp_gateway"})


# Mount the FastMCP app at /mcp. Callers POST JSON-RPC payloads here;
# FastMCP handles initialisation handshake, session IDs, and the
# tools/list + tools/call methods.
app.mount("/mcp", mcp_asgi)


# ----------------------------------------------------------------------------
# Local-run entrypoint. On Cloud Run we use uvicorn via CMD in Dockerfile
# so this block only runs when you `python server.py` directly.
# ----------------------------------------------------------------------------

def main() -> None:
    """Start uvicorn serving the composed FastAPI+FastMCP app."""
    import uvicorn

    port = int(os.getenv("PORT", "8080"))
    log.info("mcp_gateway_starting port=%d", port)
    uvicorn.run(app, host="0.0.0.0", port=port)


if __name__ == "__main__":
    main()
''')

    write_file(f'{MG}/tools/__init__.py', '''"""MCP tools package.

Each submodule in this package contributes one or more tools that get
registered with the FastMCP server in ``server.py``. Keeping tools in
their own modules makes the server file readable and lets you add a new
tool by dropping a new file in here.

See ``../ADD_YOUR_OWN_TOOL.md`` for a step-by-step walkthrough.
"""
''')

    write_file(f'{MG}/tools/gcp_project_info.py', '''"""Example MCP tool: return metadata about the current GCP project.

This is the tool shipped out-of-the-box so you can verify the MCP
gateway works end-to-end from Claude Code before you add anything of
your own. It's also a template for tools that need to call Google Cloud
APIs using the server's service-account identity.

What it returns:
    * ``project_id``    — the project the Cloud Run container is in.
    * ``project_number``— numeric GCP project number (if available).
    * ``region``        — the Cloud Run region.
    * ``enabled_apis``  — count of APIs enabled in the project.

How it authenticates:
    Application Default Credentials, the same way the LLM gateway does.
    On Cloud Run, ADC resolves to the service account attached to the
    service. Locally, to whoever ran ``gcloud auth application-default
    login``.
"""

from __future__ import annotations

import os
from typing import Any

import google.auth
import google.auth.transport.requests


def get_project_info() -> dict[str, Any]:
    """Collect a small snapshot of metadata about the current GCP project.

    Returns:
        A dict with keys ``project_id``, ``project_number`` (or None),
        ``region``, and ``enabled_apis`` (int count or None on error).

    Raises:
        Nothing that isn't caught and surfaced as a field in the result.
        MCP tools should never raise unexpectedly — they surface errors
        in the return value so the model can react usefully.
    """
    # ``google.auth.default`` returns a credentials object AND the project
    # it resolved for this call. That's the cheapest way to learn the
    # ambient project without a Service Usage API call.
    try:
        creds, project_id = google.auth.default(
            scopes=["https://www.googleapis.com/auth/cloud-platform"]
        )
    except Exception as exc:  # noqa: BLE001 — surface any auth problem verbatim
        return {
            "error": "credentials_unavailable",
            "detail": str(exc),
        }

    # The region the Cloud Run container is running in. Set by the
    # platform at runtime; falls back to "unknown" for local dev.
    region = os.getenv("GOOGLE_CLOUD_REGION") or os.getenv("K_SERVICE_REGION", "unknown")

    # Count enabled APIs via Service Usage. This requires
    # ``roles/serviceusage.serviceUsageViewer`` on the project; if absent
    # we still return the rest of the info rather than failing hard.
    enabled_apis: int | None = None
    project_number: str | None = None
    try:
        # Lazy-refresh the token. google-auth no-ops if still valid.
        if not creds.valid:
            creds.refresh(google.auth.transport.requests.Request())

        # Hand-rolled HTTP call keeps the dependency set small — no
        # extra google-cloud-* package required.
        import urllib.request
        import json as _json

        # Describe the project to get its numeric project number.
        req = urllib.request.Request(
            f"https://cloudresourcemanager.googleapis.com/v1/projects/{project_id}",
            headers={"Authorization": f"Bearer {creds.token}"},
        )
        with urllib.request.urlopen(req, timeout=5) as resp:
            data = _json.loads(resp.read())
            project_number = data.get("projectNumber")

        # Enumerate enabled services. Paginated; for the
        # "give me a count" use case, one page is enough to prove the
        # connection works — we cap at 500 for readability.
        req = urllib.request.Request(
            f"https://serviceusage.googleapis.com/v1/projects/{project_id}/services"
            "?filter=state:ENABLED&pageSize=500",
            headers={"Authorization": f"Bearer {creds.token}"},
        )
        with urllib.request.urlopen(req, timeout=5) as resp:
            data = _json.loads(resp.read())
            enabled_apis = len(data.get("services", []))
    except Exception as exc:  # noqa: BLE001 — same rationale as above
        return {
            "project_id": project_id,
            "project_number": project_number,
            "region": region,
            "enabled_apis": None,
            "warning": f"could not enumerate APIs: {exc!s}",
        }

    return {
        "project_id": project_id,
        "project_number": project_number,
        "region": region,
        "enabled_apis": enabled_apis,
    }
''')

    print(f'MCP Gateway source written to {MG}')
else:
    print('MCP gateway disabled, skipping.')

In [ ]:
if ENABLE_MCP:
    SA_ID = 'mcp-gateway'
    SA_EMAIL = f'{SA_ID}@{PROJECT_ID}.iam.gserviceaccount.com'
    SERVICE = 'mcp-gateway'
    IMAGE = f'{CR_REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}/{SERVICE}:{IMAGE_TAG}'

    # Create service account (idempotent)
    sa_exists = subprocess.run(
        f'gcloud iam service-accounts describe {SA_EMAIL} --project {PROJECT_ID}',
        shell=True, capture_output=True).returncode == 0
    if not sa_exists:
        sh(f'gcloud iam service-accounts create {SA_ID} '
           f'--project {PROJECT_ID} '
           f'--display-name="MCP Gateway (Claude Code tools)"')

    # Grant IAM roles to SA
    for role in ['roles/logging.logWriter']:
        sh(f'gcloud projects add-iam-policy-binding {PROJECT_ID} '
           f'--member="serviceAccount:{SA_EMAIL}" '
           f'--role={role} --condition=None --quiet',
           check=False)

    # Build image
    sh(f'gcloud builds submit {WORK_DIR}/mcp-gateway '
       f'--project {PROJECT_ID} --tag {IMAGE} --region {CR_REGION}')

    # Deploy to Cloud Run
    sh(f'gcloud run deploy {SERVICE} '
       f'--project {PROJECT_ID} --region {CR_REGION} '
       f'--image {IMAGE} '
       f'--service-account {SA_EMAIL} '
       f'--ingress internal-and-cloud-load-balancing '
       f'--no-allow-unauthenticated '
       f'--min-instances 0 --max-instances 5 '
       f'--cpu 1 --memory 512Mi --port 8080 '
       f'--set-env-vars GOOGLE_CLOUD_PROJECT={PROJECT_ID} '
       f'--quiet')

    # Grant invoker to principals
    principals = [p.strip() for p in ALLOWED_PRINCIPALS.split(',') if p.strip()]
    for p in principals:
        sh(f'gcloud run services add-iam-policy-binding {SERVICE} '
           f'--project {PROJECT_ID} --region {CR_REGION} '
           f'--member="{p}" --role="roles/run.invoker" --quiet',
           check=False)

    MCP_GATEWAY_URL = get_service_url(SERVICE)
    print(f'\nMCP Gateway deployed: {MCP_GATEWAY_URL}')
else:
    MCP_GATEWAY_URL = ''
    print('MCP gateway disabled, skipping.')

## 6. Deploy the Dev Portal

The Dev Portal is a static site served by nginx. It shows developers
how to connect their Claude Code install to this deployment.

The HTML contains placeholders (`__LLM_GATEWAY_URL__`, etc.) that are
replaced with the actual gateway URLs before building the image.

In [ ]:
if ENABLE_PORTAL:
    DP = f'{WORK_DIR}/dev-portal'

    # Look up gateway URLs if not already set
    if not LLM_GATEWAY_URL:
        LLM_GATEWAY_URL = get_service_url('llm-gateway')
    if not MCP_GATEWAY_URL:
        MCP_GATEWAY_URL = get_service_url('mcp-gateway')

    write_file(f'{DP}/Dockerfile', '''# =============================================================================
# Dev Portal — nginx-served static site.
#
# Cloud Run requires the container to listen on $PORT. nginx needs that
# baked into its config, so we use the /etc/nginx/templates mechanism
# from the official nginx image: any file in that dir is envsubst'd at
# startup and dropped into /etc/nginx/conf.d/.
#
# The Docker image is tiny (~10 MB) and has no runtime state.
# =============================================================================

FROM nginx:1.27-alpine

# Remove the default server block so ours is the only one.
RUN rm -f /etc/nginx/conf.d/default.conf

# Write a template config. nginx:alpine's entrypoint (docker-entrypoint.sh)
# runs envsubst over files in /etc/nginx/templates/ on startup, so $PORT
# from Cloud Run is honored without any wrapper script.
#
# IMPORTANT: The envsubst is limited to $NGINX_ENVSUBST_TEMPLATE_VARS by
# default (all env vars). We only reference $PORT.
#
# We use a shell heredoc inside RUN (not a Dockerfile COPY heredoc) for
# compatibility with builders that don't support BuildKit.
RUN mkdir -p /etc/nginx/templates && \\
    cat > /etc/nginx/templates/default.conf.template << 'NGINX_CONF'
server {
    listen       ${PORT};
    server_name  _;
    server_tokens off;
    root   /usr/share/nginx/html;
    index  index.html;
    location / {
        try_files $uri $uri/ /index.html;
    }
    location = /healthz {
        access_log off;
        return 200 'ok';
        add_header Content-Type text/plain;
    }
}
NGINX_CONF

# Copy the static site. The deploy script runs envsubst over
# index.html BEFORE building the image, so URL placeholders
# (__LLM_GATEWAY_URL__, etc.) are already substituted by this point.
COPY public/ /usr/share/nginx/html/

# Cloud Run honors whatever $PORT we're told at runtime. Default to
# 8080 for local docker runs.
ENV PORT=8080
EXPOSE 8080
''')

    write_file(f'{DP}/public/styles.css', '''/* -------------------------------------------------------------------------
 * Dev Portal styles.
 *
 * No framework, no build step. A single CSS file keeps the portal fast
 * to load and trivial to audit. Colors follow Google Cloud's own palette
 * (blues + neutrals) so the page visually belongs in the Cloud ecosystem.
 * ------------------------------------------------------------------------- */

/* Reset-ish base. */
* { box-sizing: border-box; }
html, body { margin: 0; padding: 0; }

body {
  font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
               Helvetica, Arial, sans-serif;
  color: #202124;
  background: #f8f9fa;
  line-height: 1.5;
}

.wrap {
  max-width: 820px;
  margin: 0 auto;
  padding: 48px 24px 96px;
}

header h1 {
  font-size: 2rem;
  margin: 0 0 8px;
  color: #1a73e8; /* Google blue */
}

.lede {
  font-size: 1.05rem;
  margin: 0 0 32px;
  color: #3c4043;
}

/* Card panels. */
.card {
  background: #fff;
  border: 1px solid #dadce0;
  border-radius: 8px;
  padding: 24px;
  margin-bottom: 24px;
  box-shadow: 0 1px 2px rgba(60, 64, 67, 0.08);
}

.card h2 {
  margin-top: 0;
  font-size: 1.2rem;
  color: #202124;
}

/* Key/value pairs used for the deployment summary. */
.kv {
  display: grid;
  grid-template-columns: max-content 1fr;
  column-gap: 16px;
  row-gap: 8px;
  margin: 0;
}
.kv dt { color: #5f6368; font-weight: 500; }
.kv dd { margin: 0; word-break: break-all; }

/* Code and pre blocks. */
code {
  font-family: "SF Mono", Menlo, Consolas, "Liberation Mono", monospace;
  background: #f1f3f4;
  padding: 2px 6px;
  border-radius: 4px;
  font-size: 0.9em;
}
pre {
  background: #202124;
  color: #e8eaed;
  padding: 16px;
  border-radius: 6px;
  overflow-x: auto;
  font-size: 0.9rem;
}
pre code {
  background: transparent;
  color: inherit;
  padding: 0;
  border-radius: 0;
}

/* Tabs. */
.tabs {
  display: flex;
  gap: 4px;
  border-bottom: 1px solid #dadce0;
  margin: 16px 0;
}
.tab {
  background: none;
  border: none;
  padding: 10px 16px;
  font-size: 0.95rem;
  color: #5f6368;
  cursor: pointer;
  border-bottom: 2px solid transparent;
  transition: color 0.15s, border-color 0.15s;
}
.tab:hover { color: #1a73e8; }
.tab.active {
  color: #1a73e8;
  border-color: #1a73e8;
  font-weight: 600;
}

.panel { display: none; }
.panel.active { display: block; }

/* Lists inside panels. */
.panel ol {
  padding-left: 20px;
}
.panel li { margin-bottom: 12px; }
.panel .hint {
  color: #5f6368;
  font-size: 0.9rem;
  margin: 8px 0 0;
}

/* Links. */
a { color: #1a73e8; text-decoration: none; }
a:hover { text-decoration: underline; }

/* Narrow screens. */
@media (max-width: 560px) {
  .wrap { padding: 24px 16px 48px; }
  .kv { grid-template-columns: 1fr; row-gap: 2px; }
  .kv dt { color: #5f6368; font-size: 0.85rem; }
  .kv dd { margin-bottom: 8px; }
  header h1 { font-size: 1.5rem; }
}
''')

    # Substitute real URLs into the HTML template
    html = '''<!doctype html>
<!--
  Claude Code on Vertex — Dev Portal
  ----------------------------------
  This is a static single-page welcome that tells a developer how to
  connect their Claude Code install to THIS deployment. The deploy
  script substitutes the real Cloud Run URLs into the __...__
  placeholders below before building the image.

  Substitution happens via envsubst in scripts/deploy-dev-portal.sh:
     __LLM_GATEWAY_URL__   -> https://llm-gateway-abc-uc.a.run.app
     __MCP_GATEWAY_URL__   -> https://mcp-gateway-abc-uc.a.run.app
     __PROJECT_ID__        -> my-gcp-project
     __REGION__            -> global (or us-east5, etc.)

  The static hosting is nginx inside a Cloud Run container (see the
  Dockerfile in the parent directory). No JavaScript frameworks; just
  hand-written vanilla JS for the tab switcher.
-->
<html lang="en">
  <head>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1" />
    <title>Claude Code on Vertex — Dev Portal</title>
    <link rel="stylesheet" href="styles.css" />
  </head>
  <body>
    <main class="wrap">
      <header>
        <h1>Claude Code on Vertex AI</h1>
        <p class="lede">
          Welcome. This page has everything you need to point your
          Claude Code install at
          <strong>your team's Vertex-AI-routed deployment</strong>.
          All model calls flow through the gateway below — no traffic
          to <code>api.anthropic.com</code>.
        </p>
      </header>

      <!--
        Deployment coordinates. These get substituted at deploy time
        via envsubst. If you see the raw placeholder text in your
        browser, the substitution didn't run — re-run
        scripts/deploy-dev-portal.sh.
      -->
      <section class="card">
        <h2>Your deployment</h2>
        <dl class="kv">
          <dt>GCP project</dt>
          <dd><code>__PROJECT_ID__</code></dd>
          <dt>Vertex region</dt>
          <dd><code>__REGION__</code></dd>
          <dt>LLM gateway</dt>
          <dd><code>__LLM_GATEWAY_URL__</code></dd>
          <dt>MCP gateway</dt>
          <dd><code>__MCP_GATEWAY_URL__</code></dd>
        </dl>
      </section>

      <!--
        Setup instructions — OS tabs. JS below switches the visible
        panel. Anything works without JS too; the first tab is shown
        by default and users on a broken JS environment can scroll.
      -->
      <section class="card">
        <h2>Get set up</h2>
        <p>
          Pick your OS, then paste the three commands. The script
          installs Claude Code (if missing), runs
          <code>gcloud auth application-default login</code>, and
          writes your <code>~/.claude/settings.json</code>.
        </p>

        <div class="tabs" role="tablist">
          <button class="tab active" role="tab" data-target="tab-macos">macOS</button>
          <button class="tab" role="tab" data-target="tab-linux">Linux</button>
          <button class="tab" role="tab" data-target="tab-windows">Windows</button>
        </div>

        <!-- macOS panel -->
        <div id="tab-macos" class="panel active" role="tabpanel">
          <ol>
            <li>
              Install the <code>gcloud</code> CLI if you don't have it:
              <a href="https://cloud.google.com/sdk/docs/install" target="_blank" rel="noopener">cloud.google.com/sdk/docs/install</a>.
            </li>
            <li>
              Install Node.js (Claude Code needs it):
              <code>brew install node</code>
            </li>
            <li>
              Run the developer-setup script:
              <pre><code>curl -fsSL __LLM_GATEWAY_URL__/../developer-setup.sh | bash</code></pre>
              <p class="hint">
                Prefer not to pipe curl to bash? Download
                <a href="developer-setup.sh">developer-setup.sh</a>,
                inspect it, and run it.
              </p>
            </li>
          </ol>
        </div>

        <!-- Linux panel -->
        <div id="tab-linux" class="panel" role="tabpanel">
          <ol>
            <li>
              Install the <code>gcloud</code> CLI for your distro:
              <a href="https://cloud.google.com/sdk/docs/install" target="_blank" rel="noopener">cloud.google.com/sdk/docs/install</a>.
            </li>
            <li>
              Install Node.js via your package manager (e.g.
              <code>sudo apt install nodejs npm</code> on Debian/Ubuntu).
            </li>
            <li>
              Run the developer-setup script:
              <pre><code>curl -fsSL __LLM_GATEWAY_URL__/../developer-setup.sh | bash</code></pre>
              <p class="hint">
                Or download
                <a href="developer-setup.sh">developer-setup.sh</a>
                and review before executing.
              </p>
            </li>
          </ol>
        </div>

        <!-- Windows panel -->
        <div id="tab-windows" class="panel" role="tabpanel">
          <p>
            Use <strong>WSL2 (Ubuntu)</strong> for the best experience
            — Claude Code runs cleanly under WSL and mirrors the Linux
            instructions. PowerShell-only setups are not currently
            officially supported by Claude Code.
          </p>
          <ol>
            <li>
              Install WSL2 + Ubuntu:
              <code>wsl --install -d Ubuntu</code>
            </li>
            <li>Open Ubuntu.</li>
            <li>
              Follow the <strong>Linux</strong> steps above inside the
              WSL shell.
            </li>
          </ol>
        </div>
      </section>

      <!--
        Model selection. Shown for reference; developers don't have
        to edit anything here — the setup script writes the right
        values.
      -->
      <section class="card">
        <h2>What gets written to <code>~/.claude/settings.json</code></h2>
        <pre><code>{
  "env": {
    "CLAUDE_CODE_USE_VERTEX": "1",
    "CLOUD_ML_REGION": "__REGION__",
    "ANTHROPIC_VERTEX_PROJECT_ID": "__PROJECT_ID__",
    "ANTHROPIC_VERTEX_BASE_URL": "__LLM_GATEWAY_URL__",
    "CLAUDE_CODE_DISABLE_EXPERIMENTAL_BETAS": "1"
  },
  "mcpServers": {
    "gcp-tools": {
      "type": "http",
      "url": "__MCP_GATEWAY_URL__/mcp"
    }
  }
}</code></pre>
      </section>

      <section class="card">
        <h2>Having trouble?</h2>
        <ul>
          <li>
            See the repo
            <a href="https://github.com/PTA-Co-innovation-Team/Anthropic-Google-Co-Innovation/blob/main/04-best-practice-guides/claude-code-vertex-gcp/TROUBLESHOOTING.md" target="_blank" rel="noopener">TROUBLESHOOTING.md</a>
            — most issues are covered (beta-header rejection, 429s,
            IAP problems, auth).
          </li>
          <li>
            Open a ticket with your platform team and include the
            output of <code>scripts/developer-setup.sh --diagnose</code>.
          </li>
        </ul>
      </section>
    </main>

    <!--
      Tab switcher. Pure vanilla JS so this works with no toolchain.
      If you're updating this, avoid adding frameworks — the portal
      exists to be reliable and boring.
    -->
    <script>
      document.querySelectorAll('.tab').forEach((btn) => {
        btn.addEventListener('click', () => {
          document.querySelectorAll('.tab').forEach((b) => b.classList.remove('active'));
          document.querySelectorAll('.panel').forEach((p) => p.classList.remove('active'));
          btn.classList.add('active');
          document.getElementById(btn.dataset.target).classList.add('active');
        });
      });
    </script>
  </body>
</html>
'''
    html = html.replace('__LLM_GATEWAY_URL__', LLM_GATEWAY_URL)
    html = html.replace('__MCP_GATEWAY_URL__', MCP_GATEWAY_URL)
    html = html.replace('__PROJECT_ID__', PROJECT_ID)
    html = html.replace('__REGION__', REGION)
    write_file(f'{DP}/public/index.html', html)

    print(f'Dev Portal source written to {DP}')
    print(f'  LLM URL substituted: {LLM_GATEWAY_URL}')
    print(f'  MCP URL substituted: {MCP_GATEWAY_URL}')
else:
    print('Dev portal disabled, skipping.')

In [ ]:
if ENABLE_PORTAL:
    SERVICE = 'dev-portal'
    IMAGE = f'{CR_REGION}-docker.pkg.dev/{PROJECT_ID}/{AR_REPO}/{SERVICE}:{IMAGE_TAG}'

    # Build image
    sh(f'gcloud builds submit {WORK_DIR}/dev-portal '
       f'--project {PROJECT_ID} --tag {IMAGE} --region {CR_REGION}')

    # Deploy to Cloud Run (no dedicated service account)
    sh(f'gcloud run deploy {SERVICE} '
       f'--project {PROJECT_ID} --region {CR_REGION} '
       f'--image {IMAGE} '
       f'--ingress internal-and-cloud-load-balancing '
       f'--no-allow-unauthenticated '
       f'--min-instances 0 --max-instances 2 '
       f'--cpu 1 --memory 256Mi --port 8080 '
       f'--quiet')

    # Grant invoker to principals
    principals = [p.strip() for p in ALLOWED_PRINCIPALS.split(',') if p.strip()]
    for p in principals:
        sh(f'gcloud run services add-iam-policy-binding {SERVICE} '
           f'--project {PROJECT_ID} --region {CR_REGION} '
           f'--member="{p}" --role="roles/run.invoker" --quiet',
           check=False)

    DEV_PORTAL_URL = get_service_url(SERVICE)
    print(f'\nDev Portal deployed: {DEV_PORTAL_URL}')
else:
    DEV_PORTAL_URL = ''
    print('Dev portal disabled, skipping.')

## 7. (Optional) Deploy the Dev VM

A shared GCE instance with Claude Code pre-installed, reached via IAP TCP
tunneling. **No public IP.** Costs money while running (`e2-small` ~$15/month).

Skipped by default. Set `ENABLE_DEV_VM = True` in the settings cell to deploy.

In [ ]:
if ENABLE_DEV_VM:
    # Render the startup script template
    startup = '''#!/usr/bin/env bash
# -----------------------------------------------------------------------------
# Dev VM startup script (Terraform-templated).
#
# Runs once on first boot via GCE metadata startup-script. Installs:
#   * Node.js LTS (Claude Code is a Node app)
#   * Claude Code itself (the official npm package)
#   * code-server (if $install_vscode_server == "true")
#   * A system-wide /etc/claude-code/settings.json pre-configured for
#     this deployment, so any developer on OS Login gets Claude Code
#     pointing at the right gateway out of the box.
#   * Optional: an auto-shutdown systemd timer that powers the VM off
#     after $auto_shutdown_idle_hours with no SSH sessions.
#
# Templated variables (replaced by Terraform's templatefile()):
#   $${project_id}
#   $${vertex_region}
#   $${llm_gateway_url}
#   $${mcp_gateway_url}
#   $${install_vscode_server}   ("true" or "false")
#   $${auto_shutdown_idle_hours} (0 disables)
# -----------------------------------------------------------------------------

set -euo pipefail

# Log everything to a dedicated file so operators can debug a failed
# bootstrap without re-running the startup script.
exec > >(tee -a /var/log/claude-code-bootstrap.log) 2>&1
echo "[bootstrap] starting at $(date -u +%FT%TZ)"

# --- Base OS update ----------------------------------------------------------
export DEBIAN_FRONTEND=noninteractive
apt-get update -y
apt-get install -y --no-install-recommends \\
  ca-certificates curl gnupg jq python3 python3-pip git

# --- Node.js LTS via NodeSource --------------------------------------------
if ! command -v node >/dev/null; then
  echo "[bootstrap] installing Node.js LTS"
  curl -fsSL https://deb.nodesource.com/setup_lts.x | bash -
  apt-get install -y nodejs
fi

# --- Claude Code -------------------------------------------------------------
# The official npm package. Installs the `claude` CLI globally so every
# OS Login user on the box picks it up.
if ! command -v claude >/dev/null; then
  echo "[bootstrap] installing Claude Code"
  npm install -g @anthropic-ai/claude-code
fi

# --- System-wide Claude Code settings ---------------------------------------
# Claude Code reads ~/.claude/settings.json per-user. We ALSO write a
# system-wide /etc/claude-code/settings.json and a shell snippet in
# /etc/profile.d/ that symlinks it into new users' home on first login.
install -d -m 0755 /etc/claude-code
cat >/etc/claude-code/settings.json <<'JSON'
{
  "env": {
    "CLAUDE_CODE_USE_VERTEX": "1",
    "CLOUD_ML_REGION": "${vertex_region}",
    "ANTHROPIC_VERTEX_PROJECT_ID": "${project_id}",
    "ANTHROPIC_VERTEX_BASE_URL": "${llm_gateway_url}",
    "CLAUDE_CODE_DISABLE_EXPERIMENTAL_BETAS": "1"
  },
  "mcpServers": {
    "gcp-tools": {
      "type": "http",
      "url": "${mcp_gateway_url}/mcp"
    }
  }
}
JSON

cat >/etc/profile.d/claude-code-settings.sh <<'SH'
# Link the shared Claude Code settings into the user's home on first
# login, if they don't already have a file there.
if [ -n "$HOME" ] && [ ! -e "$HOME/.claude/settings.json" ]; then
  mkdir -p "$HOME/.claude"
  ln -sf /etc/claude-code/settings.json "$HOME/.claude/settings.json"
fi
SH
chmod 0755 /etc/profile.d/claude-code-settings.sh

# --- code-server (optional) --------------------------------------------------
if [ "${install_vscode_server}" = "true" ] && ! command -v code-server >/dev/null; then
  echo "[bootstrap] installing code-server"
  curl -fsSL https://code-server.dev/install.sh | sh

  # Run code-server as a systemd user service; we bind to 0.0.0.0:8080
  # inside the VPC and expose to users via IAP's TCP tunnel.
  cat >/etc/systemd/system/code-server.service <<'UNIT'
[Unit]
Description=code-server (VS Code in the browser)
After=network-online.target

[Service]
Type=simple
# Runs as the "claude-shared" user when in shared mode — created lazily
# below. Per-user mode would set User=%i on a templated unit instead.
User=claude-shared
Environment=PASSWORD=disabled
ExecStart=/usr/bin/code-server --bind-addr 0.0.0.0:8080 --auth none
Restart=on-failure

[Install]
WantedBy=multi-user.target
UNIT

  # Create a shared user for code-server ONLY. Developers SSH in as
  # themselves via OS Login; code-server runs as this service account
  # for convenience.
  id claude-shared >/dev/null 2>&1 || useradd -m -s /bin/bash claude-shared

  systemctl daemon-reload
  systemctl enable --now code-server.service
fi

# --- Auto-shutdown timer -----------------------------------------------------
if [ "${auto_shutdown_idle_hours}" != "0" ]; then
  echo "[bootstrap] installing auto-shutdown timer (${auto_shutdown_idle_hours}h idle)"

  cat >/usr/local/sbin/claude-code-maybe-shutdown <<'SCRIPT'
#!/usr/bin/env bash
# Power off if no SSH sessions have been active within the last N hours.
IDLE_HOURS="$1"
# `who` lists active terminals; if empty, no users logged in.
if ! who | grep -q .; then
  # Also check the last SSH session end time from wtmp; be conservative.
  LAST_LOGIN_EPOCH=$(last -1 -F | head -n1 | awk '{print $(NF-1),$NF}' | xargs -I{} date -d "{}" +%s 2>/dev/null || echo 0)
  NOW_EPOCH=$(date +%s)
  if [ "$((NOW_EPOCH - LAST_LOGIN_EPOCH))" -gt "$((IDLE_HOURS * 3600))" ]; then
    logger -t claude-code-auto-shutdown "shutting down, idle > $${IDLE_HOURS}h"
    shutdown -h now
  fi
fi
SCRIPT
  chmod +x /usr/local/sbin/claude-code-maybe-shutdown

  cat >/etc/systemd/system/claude-code-auto-shutdown.service <<UNIT
[Unit]
Description=Shut down if idle > ${auto_shutdown_idle_hours}h

[Service]
Type=oneshot
ExecStart=/usr/local/sbin/claude-code-maybe-shutdown ${auto_shutdown_idle_hours}
UNIT

  cat >/etc/systemd/system/claude-code-auto-shutdown.timer <<'UNIT'
[Unit]
Description=Periodically check idle state

[Timer]
OnBootSec=15min
OnUnitActiveSec=15min

[Install]
WantedBy=timers.target
UNIT

  systemctl daemon-reload
  systemctl enable --now claude-code-auto-shutdown.timer
fi

echo "[bootstrap] done at $(date -u +%FT%TZ)"
'''
    startup = startup.replace('${project_id}', PROJECT_ID)
    startup = startup.replace('${vertex_region}', REGION)
    startup = startup.replace('${llm_gateway_url}', LLM_GATEWAY_URL or get_service_url('llm-gateway'))
    startup = startup.replace('${mcp_gateway_url}', MCP_GATEWAY_URL or get_service_url('mcp-gateway'))
    startup = startup.replace('${install_vscode_server}', 'true')
    startup = startup.replace('${auto_shutdown_idle_hours}', '2')
    # Fix Terraform double-dollar escapes
    startup = startup.replace('$${', '${')

    startup_path = f'{WORK_DIR}/startup.sh'
    write_file(startup_path, startup)
    print(f'Startup script written to {startup_path}')
else:
    print('Dev VM disabled, skipping.')

In [ ]:
if ENABLE_DEV_VM:
    SA_ID = 'claude-code-dev-vm'
    SA_EMAIL = f'{SA_ID}@{PROJECT_ID}.iam.gserviceaccount.com'
    VM_NAME = 'claude-code-dev-shared'
    ZONE = f'{FALLBACK_REGION}-a'

    # Create service account (idempotent)
    sa_exists = subprocess.run(
        f'gcloud iam service-accounts describe {SA_EMAIL} --project {PROJECT_ID}',
        shell=True, capture_output=True).returncode == 0
    if not sa_exists:
        sh(f'gcloud iam service-accounts create {SA_ID} '
           f'--project {PROJECT_ID} '
           f'--display-name="Claude Code Dev VM"')

    # Grant IAM roles to SA
    for role in ['roles/aiplatform.user', 'roles/logging.logWriter']:
        sh(f'gcloud projects add-iam-policy-binding {PROJECT_ID} '
           f'--member="serviceAccount:{SA_EMAIL}" '
           f'--role={role} --condition=None --quiet',
           check=False)

    # Create IAP firewall rule (idempotent)
    fw_exists = subprocess.run(
        f'gcloud compute firewall-rules describe allow-iap-ssh --project {PROJECT_ID}',
        shell=True, capture_output=True).returncode == 0
    if not fw_exists:
        sh(f'gcloud compute firewall-rules create allow-iap-ssh '
           f'--project {PROJECT_ID} '
           f'--direction=INGRESS --action=ALLOW '
           f'--rules=tcp:22 --source-ranges=35.235.240.0/20 '
           f'--target-tags=claude-code-dev-vm')

    # Create GCE instance (idempotent)
    vm_exists = subprocess.run(
        f'gcloud compute instances describe {VM_NAME} --project {PROJECT_ID} --zone {ZONE}',
        shell=True, capture_output=True).returncode == 0
    if not vm_exists:
        sh(f'gcloud compute instances create {VM_NAME} '
           f'--project {PROJECT_ID} --zone {ZONE} '
           f'--machine-type e2-small '
           f'--image-family debian-12 --image-project debian-cloud '
           f'--boot-disk-size 30GB '
           f'--service-account {SA_EMAIL} --scopes cloud-platform '
           f'--no-address --tags claude-code-dev-vm '
           f'--metadata enable-oslogin=TRUE '
           f'--metadata-from-file startup-script={WORK_DIR}/startup.sh '
           f'--shielded-secure-boot --shielded-vtpm --shielded-integrity-monitoring')
    else:
        print(f'VM {VM_NAME} already exists.')

    # Grant IAP + OS Login to principals
    principals = [p.strip() for p in ALLOWED_PRINCIPALS.split(',') if p.strip()]
    for p in principals:
        for role in ['roles/iap.tunnelResourceAccessor', 'roles/compute.osLogin']:
            sh(f'gcloud projects add-iam-policy-binding {PROJECT_ID} '
               f'--member="{p}" --role={role} --condition=None --quiet',
               check=False)

    print(f'\nDev VM deployed: {VM_NAME} in {ZONE}')
    print(f'SSH: gcloud compute ssh --tunnel-through-iap --project={PROJECT_ID} --zone={ZONE} {VM_NAME}')
else:
    print('Dev VM disabled, skipping.')

## 8. Deployment summary

In [ ]:
print('=' * 60)
print('  Deployment Summary')
print('=' * 60)
print(f'  Project:       {PROJECT_ID}')
print(f'  Vertex region: {REGION}')
print(f'  Cloud Run:     {CR_REGION}')
print()

if ENABLE_LLM:
    url = LLM_GATEWAY_URL or get_service_url('llm-gateway')
    print(f'  LLM Gateway:   {url}')
if ENABLE_MCP:
    url = MCP_GATEWAY_URL or get_service_url('mcp-gateway')
    print(f'  MCP Gateway:   {url}')
if ENABLE_PORTAL:
    url = DEV_PORTAL_URL or get_service_url('dev-portal')
    print(f'  Dev Portal:    {url}')
if ENABLE_DEV_VM:
    print(f'  Dev VM SSH:    gcloud compute ssh --tunnel-through-iap '
          f'--project={PROJECT_ID} --zone={FALLBACK_REGION}-a claude-code-dev-shared')

print()
print('  Developer settings.json:')
llm_url = LLM_GATEWAY_URL or get_service_url('llm-gateway') or '<llm-gateway-url>'
mcp_url = MCP_GATEWAY_URL or get_service_url('mcp-gateway') or '<mcp-gateway-url>'
print(f"""
  {{
    "env": {{
      "CLAUDE_CODE_USE_VERTEX": "1",
      "CLOUD_ML_REGION": "{REGION}",
      "ANTHROPIC_VERTEX_PROJECT_ID": "{PROJECT_ID}",
      "ANTHROPIC_VERTEX_BASE_URL": "{llm_url}",
      "CLAUDE_CODE_DISABLE_EXPERIMENTAL_BETAS": "1"
    }},
    "mcpServers": {{
      "gcp-tools": {{
        "type": "http",
        "url": "{mcp_url}/mcp"
      }}
    }}
  }}
""")
print('=' * 60)

## Next steps

Share the **Dev Portal URL** with your developers. It has copy-paste
setup instructions for macOS, Linux, and Windows (WSL2).

Or have them run on their laptop:

```bash
gcloud auth application-default login
```

Then create `~/.claude/settings.json` with the JSON block shown above
and run `claude`.

See [TROUBLESHOOTING.md](../../04-best-practice-guides/claude-code-vertex-gcp/TROUBLESHOOTING.md) for common issues.

## 9. (Optional) Tear down

**Warning:** This deletes all deployed resources. Uncomment the lines
you want to run.

In [ ]:
# Uncomment to tear down. Each line is independent.
#
# sh(f'gcloud run services delete llm-gateway --project {PROJECT_ID} --region {CR_REGION} --quiet')
# sh(f'gcloud run services delete mcp-gateway --project {PROJECT_ID} --region {CR_REGION} --quiet')
# sh(f'gcloud run services delete dev-portal --project {PROJECT_ID} --region {CR_REGION} --quiet')
# sh(f'gcloud compute instances delete claude-code-dev-shared --project {PROJECT_ID} --zone {FALLBACK_REGION}-a --quiet')
# sh(f'gcloud compute firewall-rules delete allow-iap-ssh --project {PROJECT_ID} --quiet')
# sh(f'gcloud iam service-accounts delete llm-gateway@{PROJECT_ID}.iam.gserviceaccount.com --project {PROJECT_ID} --quiet')
# sh(f'gcloud iam service-accounts delete mcp-gateway@{PROJECT_ID}.iam.gserviceaccount.com --project {PROJECT_ID} --quiet')
# sh(f'gcloud iam service-accounts delete claude-code-dev-vm@{PROJECT_ID}.iam.gserviceaccount.com --project {PROJECT_ID} --quiet')
# sh(f'gcloud artifacts repositories delete {AR_REPO} --project {PROJECT_ID} --location {CR_REGION} --quiet')
print('Teardown cell is commented out by default. Uncomment lines to run.')

## 10. Clean up temp files

Removes the temporary build directory. Safe to run at any time after
deployment is complete.

In [ ]:
shutil.rmtree(WORK_DIR, ignore_errors=True)
print(f'Cleaned up {WORK_DIR}')